# XMCA-Net-Lite: 03 - LLaVA-OneVision Transfer Answer Generation
### Purpose
Generate answers for transfer evaluation using frozen compact `LLaVA-OneVision` loaded in 4-bit NF4 quantization.
Maintains identical sharding (<= 250 items) and `completed_ids.json` checkpointing.


In [ ]:
import sys, json
from pathlib import Path
# Robust project root discovery (supports Kaggle uppercase/lowercase and local)
PROJECT_ROOT = None
for candidate in [Path.cwd(), Path.cwd().parent, Path("/kaggle/working/XMCA"), Path("/kaggle/working/xmca")]:
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate.resolve()
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, set_seed
from src.generation.llava import LLaVAOneVisionGenerator
from src.retrieval.indexer import DenseRetriever
from src.utils.logging_metadata import save_run_metadata

config = load_config()
seed = config["project"]["default_seed"]
set_seed(seed)


In [ ]:
corpora_dir = Path(config["paths"]["corpora_dir"])
retriever = DenseRetriever()
retriever.load(corpora_dir / "faiss_index.bin", corpora_dir / "corpus_metadata.json")

splits_dir = Path(config["paths"]["splits_dir"])
records_to_generate = []
for p in splits_dir.glob("*/*test.json"):
    with open(p, "r", encoding="utf-8") as f:
        records_to_generate.extend(json.load(f))

print(f"Total transfer test items: {len(records_to_generate)}")

evidence_map = {}
for item in records_to_generate:
    item_id = str(item["id"])
    top_docs = retriever.search(item["question"], top_k=config["models"]["retriever"]["top_k"])
    evidence_map[item_id] = [doc["text"] for doc, _ in top_docs]

gen_dir = Path(config["paths"]["generations_dir"]) / "llava_transfer"
generator = LLaVAOneVisionGenerator(
    model_id=config["models"]["transfer_vlm"]["model_id"],
    quantization=config["models"]["transfer_vlm"]["quantization"],
    max_new_tokens=config["models"]["transfer_vlm"]["max_new_tokens"]
)

generator.run_sharded_generation(
    records=records_to_generate,
    evidence_map=evidence_map,
    output_dir=gen_dir,
    shard_size=config["execution"]["shard_size"],
    prefix="llava_generation"
)

save_run_metadata(gen_dir, config, seed, "03_generate_llava", {"total_items": len(records_to_generate)})
print("\nStep 03 completed. LLaVA-OneVision generation shards saved.")
